# Case Study 6: Waste and Trash Detection with YOLOv8s

**Goal:** find the waste items in a photo, draw a box around each one, and name its category (Bottle, Can, Cardboard, Cup, Plastic, …).

This notebook is the **YOLOv8s** version of the YOLOv8n notebook. Everything is the same (data, epochs, image size, batch, seed) except the model, so the two can be compared fairly.

**Steps:** 1. Setup → 2. Check the dataset → 3. Train YOLOv8s → 4. Test on unseen images → 5. Error analysis → 6. Try new images

**Project folder layout** (the dataset is already in YOLO format and split into train / val / test):
```
waste-detection/
├── waste_detection_yolov8n.ipynb
├── waste_detection_yolov8s.ipynb      <- this notebook
├── yolov8n.pt
├── yolov8s.pt                         <- downloaded automatically on first run if missing
└── data/
    ├── train/  images/  labels/
    ├── val/    images/  labels/
    ├── test/   images/  labels/
    └── classes.txt
```
Select the `.venv` kernel (top-right in VS Code), then run the cells from top to bottom with **Shift + Enter**.

## 1. Setup
- **ultralytics:** trains and evaluates YOLOv8.
- **torch:** runs the neural network, on the GPU if there is one.
- **PIL:** opens images and draws on them.
- **pandas:** builds the results tables.

In [ ]:
from pathlib import Path
from collections import Counter

import pandas as pd
import torch
from PIL import Image, ImageDraw
from IPython.display import Image as Show, display
from ultralytics import YOLO

print("GPU available:", torch.cuda.is_available())

## 2. Check the dataset
**YOLO format:** every image in `images/` has a `.txt` file with the same name in `labels/`. Each line of that file is one object:
```
class_id  x_center  y_center  width  height        e.g.  2 0.51 0.43 0.22 0.35
```
The four numbers are **relative to the image size** (0 to 1). `class_id` is the line number of the class in `classes.txt`, counting from 0.

**The three splits:**
- **train:** the images the model learns from.
- **val:** checked after every epoch to keep the best model and to stop early.
- **test:** used **only once, at the end**. These are the "unseen images" the brief asks for.

The cell reads the class names, counts the images in each split, and writes a small `trash.yaml` file that tells YOLO where everything is.

In [ ]:
DATA_DIR = Path("data")
CLASSES = [c.strip() for c in (DATA_DIR / "classes.txt").read_text().splitlines() if c.strip()]
print(len(CLASSES), "classes:", CLASSES)

for split in ["train", "val", "test"]:
    n_img = len(list((DATA_DIR / split / "images").glob("*.*")))
    n_lbl = len(list((DATA_DIR / split / "labels").glob("*.txt")))
    print(f"{split:5}: {n_img} images, {n_lbl} label files")

# data config for YOLO: where the images are + the class names
DATA_YAML = DATA_DIR / "trash.yaml"
DATA_YAML.write_text(f"path: {DATA_DIR.resolve().as_posix()}\n"
                     "train: train/images\nval: val/images\ntest: test/images\n"
                     f"names: {CLASSES}\n")
print("\n" + DATA_YAML.read_text())

### Class balance
Some classes have far fewer examples than others, and rare classes usually score lower. This is why we also report results **per class**.

In [ ]:
counts = Counter()
for label_file in (DATA_DIR / "train" / "labels").glob("*.txt"):
    for line in label_file.read_text().splitlines():
        if line.strip():
            counts[CLASSES[int(line.split()[0])]] += 1

pd.Series(counts).sort_values().plot.barh(title="Objects per class (train)", figsize=(7, 4));

### Check the labels before training (good practice)
We draw the boxes from the label files onto 3 training images. If the boxes sit neatly on the objects, the labels and class names are correct.

In [ ]:
for img_path in sorted((DATA_DIR / "train" / "images").glob("*.*"))[:3]:
    img = Image.open(img_path).convert("RGB")
    draw, (W, H) = ImageDraw.Draw(img), img.size
    for line in (DATA_DIR / "train" / "labels" / f"{img_path.stem}.txt").read_text().splitlines():
        c, xc, yc, bw, bh = map(float, line.split())
        x1, y1 = (xc - bw / 2) * W, (yc - bh / 2) * H
        draw.rectangle([x1, y1, x1 + bw * W, y1 + bh * H], outline="red", width=max(2, W // 200))
        draw.text((x1, y1), CLASSES[int(c)], fill="red", font_size=max(12, W // 30))
    img.thumbnail((500, 500))
    display(img)

## 3. Train YOLOv8s
**How YOLOv8s works:** YOLO ("You Only Look Once") is a **one-stage detector**. One pass through the network predicts all boxes and classes at once, which makes it fast.
- **Backbone:** extracts features (edges → textures → object parts).
- **Neck:** combines features at 3 scales, so small, medium and large objects can all be found.
- **Head:** predicts the class and box for each object. It is *anchor-free*: it predicts box edges directly instead of adjusting preset box shapes.
- **NMS (Non-Maximum Suppression):** removes duplicate boxes on the same object and keeps the one with the highest confidence.
- "s" = small, the second-smallest YOLOv8. It has **the same layers as YOLOv8n but each layer is twice as wide** (more feature channels), giving about **11.2 M parameters** vs 3.2 M for n (≈3.5×) and about 28.6 vs 8.7 GFLOPs of compute per image.
- On COCO, YOLOv8s scores mAP@0.5:0.95 ≈ 44.9 vs ≈ 37.3 for YOLOv8n. More capacity usually means **better accuracy, especially on small or look-alike objects**, but **slower training and inference** and a bigger file.

**Training concepts:**
- **Transfer learning:** `yolov8s.pt` was already trained on **COCO** (118k images, 80 everyday objects, including *bottle* and *cup*). We fine-tune it on our 8 classes. That's why a small dataset is enough.
- **Epoch:** one pass over all training images. **Batch:** images per weight update. **imgsz:** size images are resized to.
- **Augmentation (automatic):** mosaic (4 images stitched into one), flips, colour changes and scaling. The model sees new versions of the training images in every epoch, which helps with the "different shapes, sizes and backgrounds" challenge.
- **Early stopping (`patience`):** stops training when validation mAP stops improving. The best epoch is saved as `best.pt`.
- **Loss:** box loss (how well the boxes overlap) + class loss (right category?) + DFL (sharper box edges). Training adjusts the weights to make the loss smaller.

The CPU is much slower than a GPU, so without a GPU we train for fewer epochs at a smaller image size. YOLOv8s is roughly 3× slower per epoch than YOLOv8n, so expect training to take longer.

**Fair comparison:** these settings (`EPOCHS`, `IMGSZ`, `batch`, `patience`, `seed`) are **identical to the YOLOv8n notebook**. Don't change them here unless you change them there too.

In [ ]:
GPU    = torch.cuda.is_available()
EPOCHS = 50 if GPU else 20
IMGSZ  = 640 if GPU else 416

model = YOLO("yolov8s.pt")                 # COCO-pretrained weights (auto-downloaded if missing)
model.train(
    data=str(DATA_YAML),
    epochs=EPOCHS,
    imgsz=IMGSZ,
    batch=16,
    patience=15,                            # early stopping
    seed=42,                                # reproducible results
    name="yolov8s", exist_ok=True,          # results go to runs/detect/yolov8s
)
RUN_DIR = Path(model.trainer.save_dir)      # folder where YOLO saved weights, plots and results.csv
print("Saved to:", RUN_DIR)

### Training curves
`results.png` (saved by YOLO) shows every epoch. The **losses should go down** and **precision, recall and mAP should go up**, then level off.
If validation loss starts rising while training loss keeps falling, the model is **overfitting**, i.e. memorising the training images.

In [ ]:
display(Show(filename=str(RUN_DIR / "results.png"), width=900))
display(Show(filename=str(RUN_DIR / "train_batch0.jpg"), width=600))   # a training batch after mosaic augmentation

## 4. Evaluate on the unseen TEST images
**Key terms:**
- **IoU (Intersection over Union):** overlap area ÷ combined area of the predicted and true box. A detection is counted correct if IoU ≥ 0.5 and the class is right.
- **Precision:** of all detections, how many were correct? (few false alarms)
- **Recall:** of all real objects, how many were found? (few misses)
- **AP:** area under the precision-recall curve for one class, which combines precision and recall at every confidence level.
- **mAP@0.5:** AP averaged over all classes, with IoU ≥ 0.5 counted as correct.
- **mAP@0.5:0.95:** mAP averaged over stricter IoU thresholds (0.5 to 0.95), so it rewards **tight, accurate boxes**. This measures bounding-box quality.

In [ ]:
best = YOLO(str(RUN_DIR / "weights" / "best.pt"))       # the best epoch from training
metrics = best.val(data=str(DATA_YAML), split="test", imgsz=IMGSZ,
                   plots=True,                           # also builds the confusion matrix used in step 5
                   name="yolov8s_test", exist_ok=True)
TEST_DIR = Path(metrics.save_dir)

print(f"\nPrecision    : {metrics.box.mp:.3f}")
print(f"Recall       : {metrics.box.mr:.3f}")
print(f"mAP@0.5      : {metrics.box.map50:.3f}")
print(f"mAP@0.5:0.95 : {metrics.box.map:.3f}")

### Results per class
Which classes are easy and which are hard? Small or rare items (bottle caps, cigarette butts) usually score lowest.

In [ ]:
per_class = pd.DataFrame({
    "class":     [CLASSES[i] for i in metrics.ap_class_index],
    "precision": metrics.box.p,
    "recall":    metrics.box.r,
    "AP@0.5":    metrics.box.ap50,
    "AP@0.5:0.95": metrics.box.ap,
}).round(3)
per_class.to_csv("results_per_class_yolov8s.csv", index=False)   # separate file, so the YOLOv8n results are not overwritten
per_class

## 5. Error analysis: correct, missed and false detections
YOLO's **confusion matrix** counts every detection with confidence ≥ 0.25. Rows are the **predicted** class, columns the **true** class, plus an extra **background** row and column:
- **Diagonal:** correct detections.
- **Background row:** a real object that wasn't detected (**missed**, false negative).
- **Background column:** a detection where there is no object (**false alarm**, false positive).
- **Other off-diagonal cells:** the object was found but given the **wrong class** (e.g. Plastic called Bottle).

Note: "false alarm" is counted under the class the model *predicted*. The other columns are counted under the *true* class.

In [ ]:
display(Show(filename=str(TEST_DIR / "confusion_matrix.png"), width=700))

cm = metrics.confusion_matrix.matrix       # [predicted, true]; last row/column = background
n = len(CLASSES)
errors = pd.DataFrame({
    "correct":          cm.diagonal()[:n],
    "missed":           cm[n, :n],
    "wrong class":      cm[:n, :n].sum(axis=0) - cm.diagonal()[:n],
    "false alarm":      cm[:n, n],
}, index=CLASSES).astype(int)
errors.loc["TOTAL"] = errors.sum()
errors.to_csv("error_analysis_yolov8s.csv")
errors

### See the errors on the test images
YOLO saved the **ground truth** (`labels`) and the **predictions** (`pred`) for the test images. Compare each pair: a box missing from `pred` is a missed object, and an extra box is a false alarm.

In [ ]:
for labels_img in sorted(TEST_DIR.glob("val_batch*_labels.jpg")):
    print(labels_img.name, " vs ", labels_img.name.replace("labels", "pred"))
    display(Show(filename=str(labels_img), width=520), Show(filename=str(labels_img).replace("labels", "pred"), width=520))

## 6. Try the model on new images
Put your own photos of rubbish in the `my_test_images` folder, then run this cell. It makes a good live demo for the evaluation. If the folder is empty, it runs on 3 test images instead.

In [ ]:
MY_IMAGES = Path("my_test_images")
MY_IMAGES.mkdir(exist_ok=True)
files = sorted(p for p in MY_IMAGES.iterdir() if p.suffix.lower() in {".jpg", ".jpeg", ".png"}) \
        or sorted((DATA_DIR / "test" / "images").glob("*.*"))[:3]

for r in best.predict(source=[str(f) for f in files], conf=0.25, imgsz=IMGSZ, verbose=False):
    out = Image.fromarray(r.plot()[:, :, ::-1])         # draw boxes; BGR -> RGB
    out.thumbnail((700, 700))
    print(Path(r.path).name, "->", len(r.boxes), "objects")
    display(out)

## 7. Conclusion (write this in your report)
Use the numbers above (also saved as `results_per_class_yolov8s.csv`, `error_analysis_yolov8s.csv`) together with the plots in `runs/detect/yolov8s/` (training) and `runs/detect/yolov8s_test/` (test):
1. **Overall result:** precision, recall, mAP@0.5 and mAP@0.5:0.95 on the test set.
2. **Best and worst classes:** why? (size, number of examples, look-alike classes)
3. **Error types:** mostly missed objects, false alarms or wrong classes? Relate this to the challenges in the brief (overlapping objects, different shapes, sizes and backgrounds).
4. **Limitations:** small dataset (163 original photos). The ready-made split may put augmented copies of the same photo in both train and test, which would make the scores look better than they really are.
5. **Improvements:** more images, more epochs, higher image size for small objects, tuning the confidence threshold.

### Quick viva answers
| Question | Answer |
|---|---|
| What is data leakage? | When test images (or near-copies of them, like augmented versions) are also in the training set. The model has in effect already seen them, so the scores look better than they really are. |
| Why separate validation and test sets? | Validation picks the best epoch, so we tune on it. Test is used once, which gives an honest score on unseen data. |
| What is transfer learning? | Starting from weights already trained on COCO and fine-tuning them on our images. |
| What is IoU? | Overlap ÷ union of two boxes. We count a detection as correct at IoU ≥ 0.5. |
| mAP@0.5 vs mAP@0.5:0.95? | The first checks "found it". The second also rewards tight boxes. |
| What is NMS? | It removes duplicate overlapping boxes for the same object and keeps the most confident one. |
| What does the confidence threshold do? | Higher threshold means fewer false alarms but more misses (precision ↑, recall ↓). |
| How would you deploy this? | Export `best.pt` (e.g. `best.export(format="onnx")`) and run it on a camera above the conveyor belt to trigger the right sorting bin. |